## Evaluation for Dependency Parsing

The evaluation for dependency parsing will be split up into four stages, namely:

1) Parsing the dependency outputs on the disambiguated text
2) Correcting mistakes in these outputs to create a gold standard
3) Evaluating the dependency parsing model on the gold standard


### Parsing the dependency outputs

To parse the dependency outputs, we will run the current dependency parsing model on `disambig_gold_492.txt `. The reason for running the model on the gold standard of the disambiguation, instead of raw input text, is that this allows us to create a *truly* gold standard for the dependency model. We want the gold standard for the dependencies be focused on only the dependency parsing logic, so we need the input readings to be correctly disambiguated beforehand. When evaluating the dependency set, we will be sure to check for failure cases that might have been caused by the disambiguation model, but such cases should be relatively rare. 

Using the 492 hand-corrected sentences, we will now parse the system-generated dependency outputs. We will use code from `src/booklets.py`, which will simultaneously provide us a html visualization to help with annotation of the gold standard:

In [3]:
from treebank_modules.booklets import build_dep_booklet_from_disamb
from grammar_modules.disambiguation import REPO_ROOT
from pathlib import Path

ROOT = REPO_ROOT
# Build the gold sample (Paths are from project root)
source = ROOT / "evaluation" / "eval_data" / "gold" / "disambig_gold_492.txt"
dep_cg3 = ROOT / "data" / "grammars" / "dependency.cg3"
out_conllu = ROOT / "evaluation" / "eval_data" / "sample_dep" / "opd_dep_492.conllu"
out_html = ROOT / "evaluation" / "eval_data" / "sample_dep" / "opd_dep_492.html"

# uncomment to run
build_dep_booklet_from_disamb(source, dep_cg3, out_conllu, out_html,
                            html_title="Ojibwe Dependencies on Disambiguation Gold")


Output()

Output()

Wrote booklet: /Users/matthias/labs/ELF-Lab Repos/Ojibwe_Constraint_Grammar/evaluation/eval_data/sample_dep/opd_dep_492.html  (492 sentences)
Wrote treebank: /Users/matthias/labs/ELF-Lab Repos/Ojibwe_Constraint_Grammar/evaluation/eval_data/sample_dep/opd_dep_492.conllu


### Create the gold standard

The gold standard was created manually done by editing and reassigning relationships in  `opd_dep_492.conllu`. Only currently modeled dependency relations were checked. The gold set is in `eval_data/gold/dep_gold_492.conllu`.

### Evaluating the dependency parsing module

Based on the gold set, we can now evaluate the actual performance of our dependency model.

In [4]:
from eval_modules.data_io import parse_conllu
from eval_modules.dep_eval import eval_rels_detailed, write_per_rel_table

# Data directory for evaluation
DATA = ROOT / "evaluation" / "eval_data"

# Dependency eval
# List of dep relations we focus on
FOCUS_RELS = ["nsubj","obj","iobj","det","obl","nummod","discourse","case","advmod","neg","acl:relcl","csubj","ccomp"]
gold_dep = DATA / "gold/dep_gold_492.conllu"
sys_dep  = DATA / "sample_dep/opd_dep_492.conllu"
gold_blocks = parse_conllu(gold_dep)
sys_blocks  = parse_conllu(sys_dep)
overall, per_rel = eval_rels_detailed(
    gold_blocks, sys_blocks,
    FOCUS_RELS
)
overall


{'tokens_considered': 1014,
 'precision': 0.9855875831485588,
 'recall': 0.8801980198019802,
 'f1': 0.9299163179916318,
 'accuracy': 0.8767258382642998,
 'sys_total': 902,
 'gold_total': 1010,
 'correct': 889,
 'macro_precision': 0.8402885625604767,
 'macro_recall': 0.6645326972797664,
 'macro_f1': 0.7421469112297534}

#### Eval Table: Per-Relationship Type

In [5]:
print(write_per_rel_table(per_rel, FOCUS_RELS))


rel             gold   sys  correct      P      R     F1
--------------------------------------------------------
nsubj            180   180      175  0.972  0.972  0.972
obj              191   182      176  0.967  0.921  0.944
iobj               8     4        4  1.000  0.500  0.667
det              143   139      139  1.000  0.972  0.986
obl              164   129      127  0.984  0.774  0.867
nummod             9     4        4  1.000  0.444  0.615
discourse         86    86       86  1.000  1.000  1.000
case              23    22       22  1.000  0.957  0.978
advmod           153   133      133  1.000  0.869  0.930
neg               15    15       15  1.000  1.000  1.000
acl:relcl         35     8        8  1.000  0.229  0.372
csubj              0     0        0  0.000  0.000  0.000
ccomp              3     0        0  0.000  0.000  0.000
